In [2]:
import pandas as pd
import requests
from io import BytesIO
import numpy as np

# Data Sourcing

In [3]:
def read_remote_data(url, encodings=("utf-8", "latin-1", "cp1252"), **kwargs):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    raw_bytes = response.content

    if url.lower().split("?")[0].endswith((".xlsx", ".xls")):
        return pd.read_excel(BytesIO(raw_bytes), **kwargs)

    for encoding in encodings:
        try:
            return pd.read_csv(
                BytesIO(raw_bytes),
                encoding=encoding,
                **kwargs,
            )
        except UnicodeDecodeError:
            continue

    return pd.read_csv(
        BytesIO(raw_bytes),
        encoding="latin-1",
        **kwargs,
    )

In [4]:
realtor = read_remote_data("https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_County_History.csv")
realtor_hotness = read_remote_data("https://econdata.s3-us-west-2.amazonaws.com/Reports/Hotness/RDC_Inventory_Hotness_Metrics_County_History.csv")

mortgage_rates = read_remote_data("https://www.freddiemac.com/pmms/docs/historicalweeklydata.xlsx")

fred_unemployment = read_remote_data("https://fred.stlouisfed.org/graph/fredgraph.csv?id=UNRATE")
fred_cpi = read_remote_data("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL")
fred_building_permits = read_remote_data("https://fred.stlouisfed.org/graph/fredgraph.csv?id=PERMIT")
fred_housing_starts = read_remote_data("https://fred.stlouisfed.org/graph/fredgraph.csv?id=HOUST")

census_population_2010s = read_remote_data(
    "https://www2.census.gov/programs-surveys/popest/datasets/"
    "2010-2020/counties/totals/co-est2020-alldata.csv"
)

census_population_2020s = read_remote_data(
    "https://www2.census.gov/programs-surveys/popest/datasets/"
    "2020-2025/counties/totals/co-est2025-alldata.csv"
)

# Data Cleaning

In [5]:
#remove duplicate realtor columns
common_cols = set(realtor.columns) & set(realtor_hotness.columns)
merge_keys = {"month_date_yyyymm", "county_fips"}
cols_to_drop = common_cols - merge_keys
realtor_hotness = realtor_hotness.drop(columns=list(cols_to_drop))

In [6]:
# Additional monthly economic indicators from FRED
fred_unemployment.columns = ["date", "unemployment_rate"]
fred_unemployment["date"] = pd.to_datetime(fred_unemployment["date"], errors="coerce")
fred_unemployment = fred_unemployment.dropna().copy()

fred_cpi.columns = ["date", "cpi"]
fred_cpi["date"] = pd.to_datetime(fred_cpi["date"], errors="coerce")
fred_cpi = fred_cpi.dropna().copy()

economic_indicators = pd.merge(fred_unemployment, fred_cpi, on="date", how="outer")
economic_indicators["date"] = economic_indicators["date"].dt.to_period("M").dt.to_timestamp()
economic_indicators = economic_indicators.groupby("date").mean().reset_index()

#unemployment data starts in 1948
economic_indicators = economic_indicators[economic_indicators['date']>='1948-01-01']

print(f"Null values:\n{economic_indicators.isna().sum()}")
economic_indicators.head(3)

Null values:
date                 0
unemployment_rate    0
cpi                  1
dtype: int64


,date,unemployment_rate,cpi
12,1948-01-01,3.4,23.68
13,1948-02-01,3.8,23.67
14,1948-03-01,4.0,23.50


In [7]:
def clean_census_population_long(dataframe, census_series):
    dataframe = dataframe.copy()
    dataframe.columns = dataframe.columns.str.strip().str.upper()

    # Keep county records only; Census files also contain state totals.
    if "SUMLEV" in dataframe.columns:
        dataframe = dataframe.loc[
            pd.to_numeric(dataframe["SUMLEV"], errors="coerce").eq(50)
        ].copy()

    dataframe["county_fips"] = (
        pd.to_numeric(dataframe["STATE"], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(2)
        + pd.to_numeric(dataframe["COUNTY"], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(3)
    )

    dataframe["county_name"] = dataframe["CTYNAME"].astype("string").str.strip()
    dataframe["state_name"] = dataframe["STNAME"].astype("string").str.strip()

    metric_prefixes = [
        "POPESTIMATE",
        "NPOPCHG",
        "BIRTHS",
        "DEATHS",
        "NATURALINC",
        "NATURALCHG",
        "INTERNATIONALMIG",
        "DOMESTICMIG",
        "NETMIG",
        "RINTERNATIONALMIG",
        "RDOMESTICMIG",
        "RNETMIG",
    ]

    metric_columns = [
        column
        for column in dataframe.columns
        if any(
            column.startswith(prefix)
            and column.removeprefix(prefix).isdigit()
            for prefix in metric_prefixes
        )
    ]

    census_long = dataframe.melt(
        id_vars=[
            "county_fips",
            "county_name",
            "state_name",
        ],
        value_vars=metric_columns,
        var_name="metric_year",
        value_name="value",
    )

    census_long["year"] = (
        census_long["metric_year"]
        .str.extract(r"(\d{4})$", expand=False)
        .astype("Int64")
    )

    census_long["metric"] = (
        census_long["metric_year"]
        .str.replace(r"\d{4}$", "", regex=True)
        .map(
            {
                "POPESTIMATE": "population",
                "NPOPCHG": "population_change",
                "BIRTHS": "births",
                "DEATHS": "deaths",
                "NATURALINC": "natural_increase",
                "NATURALCHG": "natural_increase",
                "INTERNATIONALMIG": "international_migration",
                "DOMESTICMIG": "domestic_migration",
                "NETMIG": "net_migration",
                "RINTERNATIONALMIG": "international_migration_rate",
                "RDOMESTICMIG": "domestic_migration_rate",
                "RNETMIG": "net_migration_rate",
            }
        )
    )

    census_long["value"] = pd.to_numeric(
        census_long["value"],
        errors="coerce",
    )

    census_long = (
        census_long.dropna(subset=["county_fips", "year", "metric"])
        .pivot_table(
            index=[
                "county_fips",
                "county_name",
                "state_name",
                "year",
            ],
            columns="metric",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .rename_axis(columns=None)
    )

    census_long["census_series"] = census_series

    return census_long.sort_values(
        ["county_fips", "year"]
    ).reset_index(drop=True)

In [8]:
census_population_2010s_long = clean_census_population_long(
    census_population_2010s,
    census_series="2010s",
)

census_population_2020s_long = clean_census_population_long(
    census_population_2020s,
    census_series="2020s",
)

In [9]:
census_population_long = pd.concat(
    [
        census_population_2010s_long.loc[
            census_population_2010s_long["year"] < 2020
        ],
        census_population_2020s_long.loc[
            census_population_2020s_long["year"] >= 2020
        ],
    ],
    ignore_index=True,
).sort_values(["county_fips", "year"]).reset_index(drop=True)

duplicate_keys = census_population_long.duplicated(
    ["county_fips", "year"],
    keep=False,
)

if duplicate_keys.any():
    raise ValueError(
        f"Found {duplicate_keys.sum()} duplicate county-year rows"
    )

print("Years:", sorted(census_population_long["year"].unique()))
print("Shape:", census_population_long.shape)
print("\nNull values:")
print(census_population_long.isna().sum())

display(census_population_long.head(3))

Years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Shape: (50294, 16)

Null values:
county_fips                         0
county_name                         0
state_name                          0
year                                0
births                              0
deaths                              0
domestic_migration                  0
domestic_migration_rate          6287
international_migration             0
international_migration_rate     6287
natural_increase                    0
net_migration                       0
net_migration_rate               6287
population                          0
census_series                       0
population_change               31430
dtype: int64


,county_fips,county_name,state_name,year,births,deaths,domestic_migration,domestic_migration_rate,international_migration,international_migration_rate,natural_increase,net_migration,net_migration_rate,population,census_series,population_change
0,01001,Autauga County,Alabama,2010,151.0,157.0,147.0,NaN,28.0,NaN,-6.0,175.0,NaN,54761.0,2010s,NaN
1,01001,Autauga County,Alabama,2011,639.0,514.0,327.0,5.945995,16.0,0.290936,125.0,343.0,6.236931,55229.0,2010s,NaN
2,01001,Autauga County,Alabama,2012,615.0,560.0,-329.0,-5.971016,0.0,0.000000,55.0,-329.0,-5.971016,54970.0,2010s,NaN


In [10]:
# Pull in monthly national supply-side indicators from FRED
fred_building_permits.columns = ["date", "building_permits"]
fred_building_permits["date"] = pd.to_datetime(fred_building_permits["date"], errors="coerce")
fred_building_permits = fred_building_permits.dropna().copy()

fred_housing_starts.columns = ["date", "housing_starts"]
fred_housing_starts["date"] = pd.to_datetime(fred_housing_starts["date"], errors="coerce")
fred_housing_starts = fred_housing_starts.dropna().copy()

supply_indicators = pd.merge(fred_building_permits, fred_housing_starts, on="date", how="outer")
supply_indicators["date"] = supply_indicators["date"].dt.to_period("M").dt.to_timestamp()
supply_indicators = supply_indicators.groupby("date").mean().reset_index()

#fred_building_permits data starts in 1960
supply_indicators = fred_building_permits[fred_building_permits['date']>='1960-01-01']

print(f"Null values:\n{supply_indicators.isna().sum()}")
supply_indicators.head(3)

Null values:
date                0
building_permits    0
dtype: int64


,date,building_permits
0,1960-01-01,1092
1,1960-02-01,1088
2,1960-03-01,955


In [11]:
mortgage_rates_df_clean = mortgage_rates.copy(deep=True)
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Unnamed: 0'].notnull()]

# take first row as header
mortgage_rates_df_clean.columns = mortgage_rates_df_clean.iloc[0].tolist()
mortgage_rates_df_clean = mortgage_rates_df_clean[1:].reset_index(drop=True)

# pull weeks from timestamps
week_parsed = pd.to_datetime(mortgage_rates_df_clean["Week"], errors="coerce")
mortgage_rates_df_clean = mortgage_rates_df_clean[week_parsed.notna()].copy()
mortgage_rates_df_clean["Week"] = week_parsed[week_parsed.notna()].dt.date

# renaming columns from base spreadsheet and dropping the shorter mortgages
mortgage_rates_df_clean.columns.values[0] = "Week"
mortgage_rates_df_clean.columns.values[1] = "U.S. 30 year FRM"
mortgage_rates_df_clean = mortgage_rates_df_clean[["Week", "U.S. 30 year FRM"]]

# truncate to recent dates
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Week']>=pd.to_datetime("2016-01-01").date()]

# move from week to month level
mortgage_rates_df_clean['Week'] = pd.to_datetime(mortgage_rates_df_clean['Week'])
mortgage_rates_df_clean['Month'] = mortgage_rates_df_clean['Week'].dt.to_period('M').dt.to_timestamp()
mortgage_rates_df_clean.drop(columns=['Week'], inplace=True)
mortgage_rates_df_clean = mortgage_rates_df_clean.groupby('Month').mean().reset_index()

print(f"Null values:\n{mortgage_rates_df_clean.isna().sum()}")
mortgage_rates_df_clean.head(3)

Null values:
Month               0
U.S. 30 year FRM    0
dtype: int64


,Month,U.S. 30 year FRM
0,2016-01-01,3.8725
1,2016-02-01,3.66
2,2016-03-01,3.694


In [12]:
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])
realtor_merge[["city", "state"]] = realtor_merge["county_name"].str.rsplit(", ", n=1, expand=True)
realtor_merge['month_date_yyyymm'] = pd.to_datetime(realtor_merge['month_date_yyyymm'], format='%Y%m')
realtor_merge['county_fips'] = realtor_merge['county_fips'].astype(str).str.zfill(5)

print(f"Null values:\n{realtor_merge.isna().sum()}")
realtor_merge.head(3)

Null values:
month_date_yyyymm                          0
county_fips                                0
county_name                                0
median_listing_price                      26
median_listing_price_mm                37537
                                       ...  
page_view_count_per_property_yy       223393
page_view_count_per_property_vs_us    203989
median_listing_price_vs_us            203989
city                                       0
state                                      0
Length: 64, dtype: int64


,month_date_yyyymm,county_fips,county_name,median_listing_price,median_listing_price_mm,median_listing_price_yy,active_listing_count,active_listing_count_mm,active_listing_count_yy,median_days_on_market,...,demand_score,median_dom_mm_day,median_dom_yy_day,median_dom_vs_us,page_view_count_per_property_mm,page_view_count_per_property_yy,page_view_count_per_property_vs_us,median_listing_price_vs_us,city,state
0,2016-07-01,01001,"autauga, al",183200.0,NaN,NaN,294,NaN,NaN,70.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,autauga,al
1,2016-07-01,01003,"baldwin, al",309900.0,NaN,NaN,2922,NaN,NaN,83.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,baldwin,al
2,2016-07-01,01005,"barbour, al",188250.0,NaN,NaN,142,NaN,NaN,130.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,barbour,al


In [13]:
processed_data_pre_model = pd.merge(
    realtor_merge,
    mortgage_rates_df_clean,
    left_on="month_date_yyyymm",
    right_on="Month",
    how="left"
)

non_realtor_columns = processed_data_pre_model.columns.difference(realtor_merge.columns)

print(
    "Null values in non-Realtor columns:\n"
    f"{processed_data_pre_model[non_realtor_columns].isna().sum()}"
)

display(processed_data_pre_model.head(3))

Null values in non-Realtor columns:
Month               0
U.S. 30 year FRM    0
dtype: int64


,month_date_yyyymm,county_fips,county_name,median_listing_price,median_listing_price_mm,median_listing_price_yy,active_listing_count,active_listing_count_mm,active_listing_count_yy,median_days_on_market,...,median_dom_yy_day,median_dom_vs_us,page_view_count_per_property_mm,page_view_count_per_property_yy,page_view_count_per_property_vs_us,median_listing_price_vs_us,city,state,Month,U.S. 30 year FRM
0,2016-07-01,01001,"autauga, al",183200.0,NaN,NaN,294,NaN,NaN,70.0,...,NaN,NaN,NaN,NaN,NaN,NaN,autauga,al,2016-07-01,3.44
1,2016-07-01,01003,"baldwin, al",309900.0,NaN,NaN,2922,NaN,NaN,83.0,...,NaN,NaN,NaN,NaN,NaN,NaN,baldwin,al,2016-07-01,3.44
2,2016-07-01,01005,"barbour, al",188250.0,NaN,NaN,142,NaN,NaN,130.0,...,NaN,NaN,NaN,NaN,NaN,NaN,barbour,al,2016-07-01,3.44


In [14]:
# 2025-10-01 was missing from the source data. forward filling to prevent data leakage
source_date = pd.Timestamp("2025-09-01")
target_date = pd.Timestamp("2025-10-01")

october_values = economic_indicators.loc[
    economic_indicators["date"].eq(source_date)
].copy()

october_values["date"] = target_date

economic_indicators = (
    pd.concat([economic_indicators, october_values], ignore_index=True)
    .drop_duplicates(subset="date", keep="last")
    .sort_values("date")
    .reset_index(drop=True)
)

display(economic_indicators.loc[
    economic_indicators["date"].isin([source_date, target_date])
])

,date,unemployment_rate,cpi
932,2025-09-01,4.4,324.245
933,2025-10-01,4.4,324.245


In [15]:
processed_data_pre_model = pd.merge(
    processed_data_pre_model,
    economic_indicators,
    left_on="month_date_yyyymm",
    right_on="date",
    how="left"
)

non_realtor_columns = processed_data_pre_model.columns.difference(realtor_merge.columns)

print(
    "Null values in non-Realtor columns:\n"
    f"{processed_data_pre_model[non_realtor_columns].isna().sum()}"
)

Null values in non-Realtor columns:
Month                   0
U.S. 30 year FRM        0
cpi                  3118
date                    0
unemployment_rate       0
dtype: int64


In [16]:
processed_data_pre_model = pd.merge(
    processed_data_pre_model,
    supply_indicators,
    left_on="month_date_yyyymm",
    right_on="date",
    how="left"
)

non_realtor_columns = processed_data_pre_model.columns.difference(realtor_merge.columns)

print(
    "Null values in non-Realtor columns:\n"
    f"{processed_data_pre_model[non_realtor_columns].isna().sum()}"
)

Null values in non-Realtor columns:
Month                   0
U.S. 30 year FRM        0
building_permits     3118
cpi                  3118
date_x                  0
date_y               3118
unemployment_rate       0
dtype: int64


In [17]:
processed_data_pre_model["year"] = processed_data_pre_model["month_date_yyyymm"].dt.year
left_keys = processed_data_pre_model[["county_fips", "year"]].copy()
right_keys = census_population_long[["county_fips", "year"]].copy()

print("Realtor years:", sorted(left_keys["year"].unique()))
print("Census years:", sorted(right_keys["year"].unique()))

join_diagnostic = left_keys.merge(
    right_keys.drop_duplicates(),
    on=["county_fips", "year"],
    how="left",
    indicator=True,
    validate="many_to_one",
)

display(
    join_diagnostic.groupby(["year", "_merge"], observed=True)
    .size()
    .unstack(fill_value=0)
)

Realtor years: [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Census years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


_merge,left_only,both
year,,
2016,54,18406
2017,108,36978
2018,108,37155
2019,108,37119
2020,0,37246
2021,0,37195
2022,0,37254
2023,0,37301
2024,0,37350


In [18]:
processed_data_pre_model["year"] = processed_data_pre_model["month_date_yyyymm"].dt.year
processed_data_pre_model = pd.merge(
    processed_data_pre_model,
    census_population_long,
    left_on=["county_fips", "year"],
    right_on=["county_fips", "year"],
    how="left"
)
processed_data_pre_model = processed_data_pre_model.drop(columns=["year"])
processed_data_pre_model.rename(columns={"month_date_yyyymm": "date"}, inplace=True)

non_realtor_columns = processed_data_pre_model.columns.difference(realtor_merge.columns)

print(
    "Null values in non-Realtor columns:\n"
    f"{processed_data_pre_model[non_realtor_columns].isna().sum()}"
)

processed_data_pre_model.head(3)

Null values in non-Realtor columns:
Month                                0
U.S. 30 year FRM                     0
births                           25273
building_permits                  3118
census_series                    25273
county_name_x                        0
county_name_y                    25273
cpi                               3118
date                                 0
date_x                               0
date_y                            3118
deaths                           25273
domestic_migration               25273
domestic_migration_rate          62519
international_migration          25273
international_migration_rate     62519
natural_increase                 25273
net_migration                    25273
net_migration_rate               62519
population                       25273
population_change               154931
state_name                       25273
unemployment_rate                    0
dtype: int64


,date,county_fips,county_name_x,median_listing_price,median_listing_price_mm,median_listing_price_yy,active_listing_count,active_listing_count_mm,active_listing_count_yy,median_days_on_market,...,domestic_migration,domestic_migration_rate,international_migration,international_migration_rate,natural_increase,net_migration,net_migration_rate,population,census_series,population_change
0,2016-07-01,01001,"autauga, al",183200.0,NaN,NaN,294,NaN,NaN,70.0,...,266.0,4.827367,7.0,0.127036,128.0,273.0,4.954403,55302.0,2010s,NaN
1,2016-07-01,01003,"baldwin, al",309900.0,NaN,NaN,2922,NaN,NaN,83.0,...,4193.0,20.409455,211.0,1.027044,265.0,4404.0,21.436499,207787.0,2010s,NaN
2,2016-07-01,01005,"barbour, al",188250.0,NaN,NaN,142,NaN,NaN,130.0,...,-492.0,-18.876611,13.0,0.498772,5.0,-479.0,-18.377839,25828.0,2010s,NaN


In [19]:
# Standardize the county-month panel before modeling
processed_data_pre_model = processed_data_pre_model.drop(
    columns=["Month", "date_x", "date_y"],
    errors="ignore"
)
processed_data_pre_model["date"] = (
    pd.to_datetime(processed_data_pre_model["date"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)
processed_data_pre_model["county_fips"] = (
    pd.to_numeric(processed_data_pre_model["county_fips"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

In [20]:
text_columns = ["county_fips", "county_name", "city", "state", "cbsa_title"]
for column in set(text_columns) & set(processed_data_pre_model.columns):
    processed_data_pre_model[column] = processed_data_pre_model[column].astype("string").str.strip()

numeric_columns = processed_data_pre_model.columns.difference(text_columns + ["date"])
processed_data_pre_model[numeric_columns] = processed_data_pre_model[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)
processed_data_pre_model[numeric_columns] = processed_data_pre_model[numeric_columns].replace(
    [np.inf, -np.inf],
    np.nan
)

In [21]:
required_keys = ["county_fips", "date"]
processed_data_pre_model = processed_data_pre_model.dropna(subset=required_keys)
valid_fips = processed_data_pre_model["county_fips"].str.fullmatch(r"\d{5}", na=False)
processed_data_pre_model = processed_data_pre_model.loc[valid_fips].copy()

In [22]:
nonnegative_columns = [
    column for column in processed_data_pre_model.columns
    if column.endswith("_count")
    or column in {
        "median_listing_price",
        "median_listing_price_per_square_foot",
        "median_square_feet",
        "average_listing_price",
        "U.S. 30 year FRM",
        "unemployment_rate",
        "cpi",
        "building_permits",
        "housing_starts",
        "population"
    }
]
for column in nonnegative_columns:
    processed_data_pre_model.loc[processed_data_pre_model[column] < 0, column] = np.nan

duplicate_keys = processed_data_pre_model.duplicated(required_keys, keep=False)
if duplicate_keys.any():
    raise ValueError(f"Found {duplicate_keys.sum()} rows with duplicate county-month keys")

In [23]:
processed_data_pre_model.drop(columns=["county_name_y","county_name_x","census_series","population_change","state_name"], errors="ignore",inplace=True)
processed_data_pre_model = processed_data_pre_model.sort_values(required_keys).reset_index(drop=True)

In [24]:
missingness_report = (
    processed_data_pre_model.isna().mean()
    .rename("missing_fraction")
    .sort_values(ascending=False)
    .to_frame()
)

feature_columns = processed_data_pre_model.columns.difference(required_keys)

missingness_by_year = (
    processed_data_pre_model[feature_columns]
    .isna()
    .assign(year=processed_data_pre_model["date"].dt.year)
    .groupby("year")
    .mean()
    .reset_index()
    .sort_values("year")
    .reset_index(drop=True)
)

# Review, final cleaning, and save

In [25]:
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None
):
    display(processed_data_pre_model.head(3))
    display(missingness_report.head(15))
    display(missingness_by_year)

,date,county_fips,median_listing_price,median_listing_price_mm,median_listing_price_yy,active_listing_count,active_listing_count_mm,active_listing_count_yy,median_days_on_market,median_days_on_market_mm,median_days_on_market_yy,new_listing_count,new_listing_count_mm,new_listing_count_yy,price_increased_count,price_increased_count_mm,price_increased_count_yy,price_increased_share,price_increased_share_mm,price_increased_share_yy,price_reduced_count,price_reduced_count_mm,price_reduced_count_yy,price_reduced_share,price_reduced_share_mm,price_reduced_share_yy,pending_listing_count,pending_listing_count_mm,pending_listing_count_yy,median_listing_price_per_square_foot,median_listing_price_per_square_foot_mm,median_listing_price_per_square_foot_yy,median_square_feet,median_square_feet_mm,median_square_feet_yy,average_listing_price,average_listing_price_mm,average_listing_price_yy,total_listing_count,total_listing_count_mm,total_listing_count_yy,pending_ratio,pending_ratio_mm,pending_ratio_yy,quality_flag,cbsa_code,cbsa_title,hh_rank,hotness_rank,hotness_rank_mm,hotness_rank_yy,hotness_score,supply_score,demand_score,median_dom_mm_day,median_dom_yy_day,median_dom_vs_us,page_view_count_per_property_mm,page_view_count_per_property_yy,page_view_count_per_property_vs_us,median_listing_price_vs_us,city,state,U.S. 30 year FRM,unemployment_rate,cpi,building_permits,births,deaths,domestic_migration,domestic_migration_rate,international_migration,international_migration_rate,natural_increase,net_migration,net_migration_rate,population
0,2016-07-01,01001,183200.0,NaN,NaN,294.0,NaN,NaN,70.0,NaN,NaN,90.0,NaN,NaN,0.0,NaN,NaN,0.0000,NaN,NaN,54.0,NaN,NaN,0.1194,NaN,NaN,71.0,NaN,NaN,94.0,NaN,NaN,1940.0,NaN,NaN,210994.0,NaN,NaN,363.0,NaN,NaN,0.2402,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,autauga,al,3.440,4.8,240.101,1198.0,675.0,547.0,266.0,4.827367,7.0,0.127036,128.0,273.0,4.954403,55302.0
1,2016-08-01,01001,177850.0,NaN,NaN,284.0,NaN,NaN,67.0,NaN,NaN,72.0,NaN,NaN,2.0,NaN,NaN,0.0045,NaN,NaN,66.0,NaN,NaN,0.1563,NaN,NaN,70.0,NaN,NaN,93.0,NaN,NaN,1876.0,NaN,NaN,201524.0,NaN,NaN,354.0,NaN,NaN,0.2447,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,autauga,al,3.435,4.9,240.545,1201.0,675.0,547.0,266.0,4.827367,7.0,0.127036,128.0,273.0,4.954403,55302.0
2,2016-09-01,01001,174900.0,NaN,NaN,275.0,NaN,NaN,74.0,NaN,NaN,70.0,NaN,NaN,2.0,NaN,NaN,0.0050,NaN,NaN,66.0,NaN,NaN,0.1636,NaN,NaN,54.0,NaN,NaN,91.0,NaN,NaN,1902.0,NaN,NaN,201240.0,NaN,NaN,329.0,NaN,NaN,0.1964,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,autauga,al,3.460,5.0,241.176,1303.0,675.0,547.0,266.0,4.827367,7.0,0.127036,128.0,273.0,4.954403,55302.0


,missing_fraction
price_increased_count_mm,0.741689
price_increased_count_yy,0.735847
cbsa_title,0.592959
cbsa_code,0.592959
page_view_count_per_property_yy,0.590011
hotness_rank_yy,0.590011
page_view_count_per_property_mm,0.543033
median_dom_yy_day,0.538763
median_dom_mm_day,0.538763
median_dom_vs_us,0.538763


,year,U.S. 30 year FRM,active_listing_count,active_listing_count_mm,active_listing_count_yy,average_listing_price,average_listing_price_mm,average_listing_price_yy,births,building_permits,cbsa_code,cbsa_title,city,cpi,deaths,demand_score,domestic_migration,domestic_migration_rate,hh_rank,hotness_rank,hotness_rank_mm,hotness_rank_yy,hotness_score,international_migration,international_migration_rate,median_days_on_market,median_days_on_market_mm,median_days_on_market_yy,median_dom_mm_day,median_dom_vs_us,median_dom_yy_day,median_listing_price,median_listing_price_mm,median_listing_price_per_square_foot,median_listing_price_per_square_foot_mm,median_listing_price_per_square_foot_yy,median_listing_price_vs_us,median_listing_price_yy,median_square_feet,median_square_feet_mm,median_square_feet_yy,natural_increase,net_migration,net_migration_rate,new_listing_count,new_listing_count_mm,new_listing_count_yy,page_view_count_per_property_mm,page_view_count_per_property_vs_us,page_view_count_per_property_yy,pending_listing_count,pending_listing_count_mm,pending_listing_count_yy,pending_ratio,pending_ratio_mm,pending_ratio_yy,population,price_increased_count,price_increased_count_mm,price_increased_count_yy,price_increased_share,price_increased_share_mm,price_increased_share_yy,price_reduced_count,price_reduced_count_mm,price_reduced_count_yy,price_reduced_share,price_reduced_share_mm,price_reduced_share_yy,quality_flag,state,supply_score,total_listing_count,total_listing_count_mm,total_listing_count_yy,unemployment_rate
0,2016,0.0,0.0,1.000000,1.000000,0.000054,1.000000,1.000000,0.002925,0.000000,1.000000,1.000000,0.0,0.000000,0.002925,1.000000,0.002925,0.002925,1.000000,1.000000,1.000000,1.000000,1.000000,0.002925,0.002925,0.001896,1.000000,1.000000,1.000000,1.000000,1.000000,0.000054,1.000000,0.001625,1.000000,1.000000,1.000000,1.000000,0.001625,1.000000,1.000000,0.002925,0.002925,0.002925,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.191495,1.000000,1.000000,0.191820,1.000000,1.000000,0.002925,0.0,1.000000,1.000000,0.000379,1.000000,1.000000,0.0,1.000000,1.000000,0.000379,1.000000,1.000000,1.00000,0.0,1.000000,0.0,1.000000,1.000000,0.0
1,2017,0.0,0.0,0.499299,0.504854,0.000000,0.499245,0.504557,0.002912,0.000000,0.807609,0.807609,0.0,0.000000,0.002912,0.781993,0.002912,0.002912,0.781993,0.781993,0.781993,1.000000,0.781993,0.002912,0.002912,0.001213,0.499838,0.505393,0.781993,0.781993,0.781993,0.000000,0.499245,0.001726,0.500108,0.505582,0.781993,0.504557,0.001726,0.500108,0.505582,0.002912,0.002912,0.002912,0.0,0.546729,0.560400,0.825595,0.781993,1.000000,0.167017,0.598042,0.625115,0.167044,0.589872,0.616702,0.002912,0.0,0.828399,0.841180,0.000566,0.499299,0.504638,0.0,0.566845,0.577792,0.000566,0.499299,0.504638,0.49822,0.0,0.781993,0.0,0.499191,0.504530,0.0
2,2018,0.0,0.0,0.001744,0.010010,0.000000,0.001449,0.009500,0.002898,0.000000,0.540456,0.540456,0.0,0.000000,0.002898,0.479269,0.002898,0.002898,0.479269,0.479269,0.479269,0.783029,0.479269,0.002898,0.002898,0.001073,0.002845,0.011486,0.479269,0.479269,0.479269,0.000000,0.001449,0.000751,0.002254,0.011432,0.479269,0.009500,0.000751,0.002254,0.011432,0.002898,0.002898,0.002898,0.0,0.097443,0.107345,0.479269,0.479269,0.783029,0.128868,0.169793,0.221990,0.129029,0.154818,0.203365,0.002898,0.0,0.650484,0.667338,0.000322,0.001744,0.009473,0.0,0.142205,0.155489,0.000322,0.001744,0.009473,0.00000,0.0,0.479269,0.0,0.001449,0.009312,0.0
3,2019,0.0,0.0,0.001558,0.005211,0.000081,0.001612,0.004916,0.002901,0.000000,0.540011,0.540011,0.0,0.000000,0.002901,0.478765,0.002901,0.002901,0.478765,0.478765,0.478765,0.478765,0.478765,0.002901,0.002901,0.001827,0.003573,0.007145,0.478765,0.478765,0.478765,0.000081,0.001612,0.000564,0.002230,0.005910,0.478765,0.004916,0.000484,0.002149,0.005829,0.002901,0.002901,0.002901,0.0,0.094367,0.095791,0.478765,0.478765,0.478765,0.107959,0.133559,0.181777,0.108013,0.130470,0.166438,0.002901,0.0,0.703898,0.650388,0.000161,0.001585,0.004970,0.0,0.146668,0.139

In [ ]:
# Remove lag features marked by an underscore-delimited "mm" or "yy" segment
# high in missingness and TFT does not require lag features
lag_columns = processed_data_pre_model.columns[
    processed_data_pre_model.columns.str.contains(
        r"(?:^|_)(?:mm|yy)(?:_|$)",
        case=False,
        regex=True,
    )
].tolist()

processed_data_pre_model = processed_data_pre_model.drop(columns=lag_columns)

print(f"Removed {len(lag_columns)} lag features:")
print(lag_columns)

In [ ]:
# Drop rows where median_listing_price is NA
processed_data_pre_model = processed_data_pre_model.dropna(subset=["median_listing_price"])

# ensure date is date
processed_data_pre_model["date"] = pd.to_datetime(processed_data_pre_model["date"], errors="coerce")

In [ ]:
processed_data_pre_model['year'] = pd.to_datetime(processed_data_pre_model['date']).dt.year
processed_data_pre_model['month'] = pd.to_datetime(processed_data_pre_model['date']).dt.month

In [30]:
from pathlib import Path

project_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "data_and_logs").is_dir()
)

output_path = (
    project_root
    / "data_and_logs"
    / "data"
    / "raw"
    / "cleaned_data.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
processed_data_pre_model.to_csv(output_path, index=False)